In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
import os

load_dotenv(override=True)
openai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

openrouter =  OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = os.getenv("OPENROUTER_API_KEY")
)

In [ ]:
response = openrouter.chat.completions.create(
    model = "inclusionai/ling-3.2-flash:free",
    messages = [
        { "role": "system", "content": "You are a helpful assistant."},
        { "role": "user", "content": "Hello! How are you?"}
    ]
).choices[0].message.content
print(response)

In [ ]:
reader = PdfReader("info.pdf")
info = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        info += text

In [ ]:
print(info)

In [ ]:
response = openai.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role" : "system", "content": "You are a helpful assistant and you are made to summarize information."},
        {"role" : "user", "content": info}
    ]
)
summary = response.choices[0].message.content


In [ ]:
with open("summary.txt","w") as f:
    f.write(summary)
    

In [ ]:
print(summary)

In [ ]:
system_prompt = f"""
# Your Role

You are an AI customer support assistant
representing TechNova Solutions.

# Company Summary

{summary}


The information above is your primary knowledge source.
Use it to provide accurate and relevant answers.

# Rules

1. Engage with customers professionally and politely.
   Be friendly, helpful and informative.

2. Only answer questions related to TechNova Solutions,
   its travel services, packages, pricing, booking procedures
   and business policies.

3. Avoid answering unrelated questions.
   Politely explain that your role is to assist with
   TechNova Solutions' travel services and guide the
   conversation back to relevant topics.

4. Always use the provided business information as your
   source of truth.

5. Never invent travel packages, prices, discounts,
   services, contact details or company policies.

6. If you don't know the answer or the information is
   unavailable in the provided context, clearly say so.
   Never make up an answer.

7. Do not guarantee flight availability, hotel availability,
   booking confirmations or visa approvals.

8. Clearly mention that package prices are indicative
   estimates when discussing sample pricing.

9. If a customer wants to get in touch with the company,
   politely ask for their email address.
   Once they provide it, the appropriate email-recording
   tool can be used to save it.

10. Never claim that an email has been saved unless the
    email-recording tool confirms successful execution.

11. Protect customer privacy. Never ask for passwords,
    OTPs, PINs or sensitive payment credentials.

12. If a customer asks whether TechNova Solutions is a real
    company, clearly explain that it is a fictional business
    created for demonstration purposes.

# Response Style

- Use simple and understandable language.
- Keep responses relevant to the customer's question.
- Be professional, friendly and conversational.
- Use bullet points when explaining multiple services,
  packages or policies.
- Avoid unnecessarily long responses.

IMPORTANT:
Always stay in character as the AI customer support
assistant of TechNova Solutions.

Your primary goal is to help customers understand
the travel services offered by TechNova Solutions
and assist them with their inquiries.


...
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
def chat(message,history):
    history = [ {"role": h["role"], "content": h["content"]} for h in history]
    response = openai.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    ).choices[0].message.content
    for h in history:
        print(f"{h['role']}: {h['content']}")
    return response


In [ ]:
message = "What are your technical skills?"

history = [
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi! How can I help you?"},
    {"role": "user", "content": "Tell me about yourself"},
    {"role": "assistant", "content": "I am an AI digital twin."}
]

response = chat(message, history)
print(response)

In [ ]:
message = "Which skills are required for the position?"

history = [
    {"role": "user", "content": "Hello"},
    {"role": "assistant", "content": "Hi! How can I help you?"},
    {"role": "user", "content": "Tell me about yourself"},
    {"role": "assistant", "content": "I am an AI digital twin."},
    {"role": "user", "content": message}
]

response = chat(message, history)
print(response)

In [ ]:
# gr.ChatInterface(chat).launch()

## Adding tools


In [ ]:
def record_email_tool(email):
    with open("emails.txt","a") as f:
        f.write(email + "\n")
    print(f"Email recorded: {email}")
    return "Email recorded successfully."

def record_phone_tool(phone):
    with open("phone_numbers.txt", "a") as f:
        f.write(phone + "\n")
    print(f"Phone number recorded: {phone}")
    return "Phone number recorded successfully."

def record_feedback_tool(feedback):
    with open("feedback.txt", "a") as f:
        f.write(feedback + "\n")
    print(f"Feedback recorded: {feedback}")
    return "Feedback recorded successfully."

In [ ]:
record_email_tool("john@example.com")

## Adding json to describe the tool.

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_phone_tool_json = {
    "name": "record_phone_tool",
    "description": "Use this tool to record that a user provided their phone number",
    "parameters": {
        "type": "object",
        "properties": {
            "phone": {
                "type": "string",
                "description": "The phone number of this user"
            }
        },
        "required": ["phone"],
        "additionalProperties": False
    }
}

record_feedback_tool_json = {
    "name": "record_feedback_tool",
    "description": "Use this tool to record feedback provided by a user",
    "parameters": {
        "type": "object",
        "properties": {
            "feedback": {
                "type": "string",
                "description": "The feedback provided by this user"
            }
        },
        "required": ["feedback"],
        "additionalProperties": False
    }
}

In [ ]:

tools = [ 
    {"type":"function", "function": record_email_tool_json},
    {"type":"function", "function": record_phone_tool_json},
    {"type":"function", "function": record_feedback_tool_json}
]

## Adding chat function

In [ ]:
system_prompt_2 = """ 
You are an AI Response Evaluator responsible for evaluating the quality and accuracy of responses generated by LLM1, an AI customer support assistant for TechNova Solutions.

Your task is to carefully evaluate LLM1's response against the customer's original question, the company knowledge base, and the rules provided below.

You must NOT answer the customer's question directly. Your only responsibility is to judge LLM1's response.

# Evaluation Criteria

Evaluate LLM1's response using the following criteria:

1. **Factual Accuracy**
   - Verify that all company-related information is supported by the provided knowledge base.
   - Check that package names, prices, durations, inclusions, services, and policies are accurate.
   - Identify any fabricated or unsupported information.

2. **Relevance**
   - Check whether the response answers the customer's actual question.
   - Ensure the response stays within TechNova Solutions' business scope.
   - Unrelated questions should be politely redirected.

3. **Policy Compliance**
   - Ensure sample package prices are clearly identified as indicative estimates.
   - Ensure LLM1 does not guarantee flight availability, hotel availability, visa approval, or booking confirmation.
   - Verify that booking confirmation is explained correctly.
   - Check that cancellations, refunds, and amendments follow the stated supplier-dependent policies.

4. **Privacy and Security**
   - Ensure LLM1 never requests passwords, OTPs, PINs, or sensitive payment credentials.
   - Ensure customer information is handled appropriately.
   - Check that an email is not claimed to be saved without successful tool execution.

5. **Fictional Company Disclosure**
   - If the customer asks whether TechNova Solutions is a real company, LLM1 must clearly explain that it is a fictional business created for demonstration purposes.

6. **Response Quality**
   - Ensure the response is polite, professional, understandable, and relevant.
   - Avoid unnecessary information and misleading statements.

# Judgment Rules

- PASS: The response is factually correct, relevant, and follows all applicable rules.
- FAIL: The response contains at least one significant factual error, unsupported claim, misleading statement, or policy violation.
- Do not mark a response as FAIL for minor wording differences or harmless variations.
- Do not penalize LLM1 for saying that information is unavailable when the knowledge base does not provide it.
- Evaluate only the rules relevant to the customer's question.
- Never assume information that is not present in the knowledge base.
- Do not reward a response simply because it sounds professional. Accuracy is more important than presentation.

# Output Format

Return your evaluation strictly in the following JSON format:

{
    "judgment": "PASS or FAIL",
    "feedback": "Explain briefly why the response passed or failed.",
    "corrected_response": "If FAIL, provide a corrected response based only on the knowledge base. If PASS, return an empty string."
}

Do not include Markdown formatting, additional explanations, or text outside the JSON object.

# Important

You are an independent evaluator. Do not blindly trust LLM1's response.

Your primary responsibility is to ensure that customers receive accurate, safe, relevant, and policy-compliant information about TechNova Solutions.

# Knowledge Grounding Rules

- Use the company information provided in the evaluation input as your only source of truth.
- Do not use pretrained knowledge or assumptions to verify company-specific claims.
- The evaluation input may contain a condensed version of the company knowledge base.
- If a particular detail is not available in the provided reference, treat it as unverified.
- LLM1 should not invent missing information. It should clearly communicate when information is unavailable.
- Do not mark a response as FAIL simply because the evaluation reference does not contain enough information to independently verify a harmless detail, provided LLM1 appropriately acknowledges the uncertainty.
- If LLM1 presents unsupported company-specific information as a confirmed fact, mark it as FAIL.
"""

In [ ]:
import json

def chat(message, history):
    # Clean Gradio's history so ONLY 'role' and 'content' remain
    clean_history = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in history
    ]
    
    messages = (
        [{"role": "system", "content": system_prompt}] 
        + clean_history 
        + [{"role": "user", "content": message}]
    )
    
    response = openai.chat.completions.create(
        model="openai/gpt-oss-120b", 
        messages=messages, 
        tools=tools
    )
         
    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        messages.append(assistant_msg.model_dump(exclude_none=True))
        
        for tool_call in assistant_msg.tool_calls:
            if tool_call.function.name == "record_email_tool":
                email = json.loads(tool_call.function.arguments).get("email")
                result = record_email_tool(email)
            elif tool_call.function.name == "record_phone_tool":
                phone = json.loads(tool_call.function.arguments).get("phone")
                result = record_phone_tool(phone)
            elif tool_call.function.name == "record_feedback_tool":
                feedback = json.loads(tool_call.function.arguments).get("feedback")
                result = record_feedback_tool(feedback)
            else:
                result = f"Unknown tool: {tool_call.function.name}"
            messages.append({
                "role": "tool",
                "content": str(result),
                "tool_call_id": tool_call.id
            })
                
        response = openai.chat.completions.create(
            model="openai/gpt-oss-120b", 
            messages=messages, 
            tools=tools
        )
            
    LLM1_response = response.choices[0].message.content
    response = openrouter.chat.completions.create(
        model="inclusionai/ling-3.2-flash:free",
        messages=[
            {"role": "system", "content": system_prompt_2},
            {"role": "user", "content": f"Customer Question: {message}\n\nLLM1 Response: {LLM1_response}"}
        ],
    )
    LLM2_response = response.choices[0].message.content

    try:
        evaluation = json.loads(LLM2_response)
        judgment = evaluation.get("judgment")
        feedback = evaluation.get("feedback")
        corrected_response = evaluation.get("corrected_response")

        if judgment == "PASS":
            final_response = LLM1_response

        elif judgment == "FAIL" and corrected_response:
            final_response = corrected_response

        else:
            final_response = (
                "I'm sorry, but I couldn't verify my response. "
                "Please contact our support team for assistance."
            )

    except (json.JSONDecodeError, TypeError):
        final_response = (
            "I'm sorry, but I'm unable to verify my response right now."
        )

    return final_response


    


In [ ]:
gr.ChatInterface(chat).launch()

In [ ]:
with open("emails.txt", "r") as f:
    recorded_emails = f.read().strip().split("\n")
    print("Recorded Emails:", recorded_emails)
with open("phone_numbers.txt", "r") as f:
    recorded_phone_numbers = f.read().strip().split("\n")
    print("Recorded Phone Numbers:", recorded_phone_numbers)
with open("feedback.txt", "r") as f:
    recorded_feedback = f.read().strip().split("\n")
    print("Recorded Feedback:", recorded_feedback)
